In [3]:
#Import Packages
import functools
import random
from copy import copy
import pettingzoo
import numpy as np
from gymnasium.spaces import Discrete, MultiDiscrete
from pettingzoo import ParallelEnv
import matplotlib.pyplot as plt

In [5]:
#Environment Definition
#Need parallel environment for this as multiple agents can act at once
from pettingzoo import ParallelEnv


class CustomEnvironment(ParallelEnv):
    metadata = {
        "name": "multi_satellite_downlink_v0",
    }
    

    def __init__(self):
        """The init method takes in environment arguments.

        Note: as of v1.18.1, the action_spaces and observation_spaces attributes are deprecated.
        Spaces should be defined in the action_space() and observation_space() methods.
        If these methods are not overridden, spaces will be inferred from self.observation_spaces/action_spaces, raising a warning.

        These attributes should not be changed after initialization.
        """
        self.master_contact_plan = None
        self.satellite_plan = None
        self.satellite_data=None
        self.timestep = None
        self.possible_agents = ["satellite1","satellite2","satellite3"]
        self.delivery_ratio=None
        self.energy_efficiency=None
        self.energy_expenditure=None
        self.delivered_packets=None
        

    def reset(self, seed=None, options=None):
        """Reset set the environment to a starting point.

        It needs to initialize the following attributes:
        - agents
        - timestamp
        - Initial Data Volume
        - Master Contact Plan
        - Number of Delivered Packets
        - Energy Expenditure
        - Delivery Ratio
        - Energy Efficiency
        - observation
        - infos

        And must set up the environment so that render(), step(), and observe() can be called without issues.
        """
        self.agents = copy(self.possible_agents)
        self.timestep = 0
        #Define the initial data volume at each satellite
        for i in range(3):
            self.satellite_data[i]=random.randint(5,100)
        print(self.satellite_data[i])
        #Define the master contact plan
        #Contains 30 contacts (total), weather values for each contact and which contacts are shared
            
            #Code below was written with a template created with ChatGPT

            # Parameters
            nodes = ['Satellite1', 'Satellite2', 'Satellite3']
            max_rows_per_node = 10
            node_usage = {node: 0 for node in nodes}
            
            matrix = []
            
            row_index = 1
            while any(count < max_rows_per_node for count in node_usage.values()):
                # Get nodes that still have availability
                available_nodes = [node for node in nodes if node_usage[node] < max_rows_per_node]
            
                # Determine how many nodes to assign in this row
                max_possible = len(available_nodes)
                num_nodes_in_row = random.randint(1, max_possible)
            
                # Select nodes for the row
                selected_nodes = random.sample(available_nodes, num_nodes_in_row)
            
                # Update usage count
                for node in selected_nodes:
                    node_usage[node] += 1
            
                # Create and store the row
                row = [random.random(), 10, selected_nodes]
                matrix.append(row)
            
                row_index += 1
            
            # Output the matrix
            for row in matrix:
                print(row)
            
            # Show final node usage
            print("\nFinal node usage:")
            for node, count in node_usage.items():
                print(f"{node}: {count}")
            
            print(f"\nTotal rows generated: {len(matrix)}")

            #Next we need either divide up the master contact plan during observations or 
            #Pad the complete matrix up to 30
            if len(matrix)<30:
                rows, cols = 30, 3
                new_matrix = [[None for _ in range(cols)] for _ in range(rows)]
                for i in range(30):
                    if i>(len(matrix)-1):
                        new_matrix[i]=[-1,-1,-1]
                    else:
                        print(i)
                        print(new_matrix)
                        new_matrix[i]=matrix[i]
            
            
            else:
            new_matrix=matrix
            print(new_matrix)


        #Next we must convert the new matrix into a format that can be loaded into the observation space effectively
        rows, cols = 30, 5
        obs_matrix = [[None for _ in range(cols)] for _ in range(rows)]
        for i in range(30):
            for j in range(3):
                if j<2:
                    obs_matrix[i][j]=new_matrix[i][j]
                elif j==2:
                    current_element=new_matrix[i][j]
                    print(current_element)
                    encoded_row=[0,0,0]
                    if current_element!=-1:
                        for k in range(len(current_element)-1):
                            if current_element[k]=="satellite1":
                                encoded_row[0]=1
                            elif current_element[k]=="satellite2":
                                encoded_row[1]=1
                            elif current_element[k]=="satellite3":
                                encoded_row[2]=1
                    else:
                        encoded_row=[0,0,0]
                    for k in range(2,5):
                        print("Encoded Row")
                        print(encoded_row)
                        obs_matrix[i][k]=encoded_row[k-2]
        print("Obs matrix")
        print(obs_matrix)
        #Next we must define the action mask 
        action_mask=[[0,0],[0,0],[0,0]]
        if obs_matrix[1][2]==1:
            action_mask[0]=[1,1]
        if obs_matrix[1][3]==1:
            action_mask[1]=[1,1]
        if obs_matrix[1][4]==1:
            action_mask[2]=[1,1]
                


        
                #randomly allocate weather conditions and if the contact is shared
        self.timestep=1
        self.master_contact_plan=obs_matrix        
        print(self.master_contact_plan)
        #Set delivery ratio for each satellite
        self.delivery_ratio=[None,None,None]
        self.delivered_packets=[0,0,0]
        #Set energy expenditure for each satellite
        self.energy_efficiency=[None,None,None]
        self.energy_expenditure=[0,0,0]
        

        #Implement observations
        #We start will full observability
        #Move to partial observability later

        
        observations = {
            a: (
                obs_matrix,
                self.delivered_packets,
                self.energy_expenditure,
                self.satellite_data,
                self.timestep,
                action_mask[a]
                #self.prisoner_x + 7 * self.prisoner_y,
                #self.guard_x + 7 * self.guard_y,
                #self.escape_x + 7 * self.escape_y,

                #Need to implement action mask
                
            )
            for a in self.agents
        }

        # Get dummy infos. Necessary for proper parallel_to_aec conversion
        infos = {a: {} for a in self.agents}

        return observations, infos

    def step(self, actions):
        """Takes in an action for the current agent (specified by agent_selection).

        Needs to update:
        - prisoner x and y coordinates
        - guard x and y coordinates
        - terminations
        - truncations
        - rewards
        - timestamp
        - infos

        And any internal state used by observe() or render()
        """
        # Execute actions
        #First we must get the actions for each satellite
        


        
        satellite_actions=actions["satellite1","satellite2","satellite3"]
        
        
        

        #Next we update the delivery and energy expenditure for all satellites
        
        
        #This logic was partially aided by ChatGPT (used to it to get basic structure of checking if satellites are competing for ground stations, however reward and majority of logic is my own)
        delivered_packets=0
        excess_energy=0
        total_ones = satellite_actions.count(1)
        #Next we check if there are any collisions between the agents
        collision_matrix=checkActionConflict(timestep,satellite_actions)


        
        for i in range(len(satellite_actions)):
            current_value = satellite_actions[i]
            other_ones_exist = (total_ones - (1 if current_value == 1 else 0)) > 0

            if collision_matrix[i]==1:
                #We have a conflict, no packets are sent and the satellite receives a large penalty
                reward[i]=-10
            else:
                if current_value==0:
                    reward[i]=0
                else:
                    delivered_packets,excess_energy=updateDeliveryandEnergy(self,agent,row[1],row[2])
                    if delivered_packets >0:
                        #Positive Reward
                        reward[i]=delivered_packets/(excess_energy+1)
                    else:
                        reward[i]=-1


        
        terminations={a:False for a in self.agents}
        #Next we must update the parameters changed in step function
        updateTimestep()
        #Next we must update the number of delivered packets and energy expenditure

        


        #Next we update the contact plan representation


        #

        
        #Outline conditions for terminations
        if timestep>30 :
        #All satellites have 
            truncations={a: True for a in self.agents}
        #Truncation conditions
        if delivered_packets[0]==satellite_data[0] && delivered_packets[1]==satellite_data[1] && delivered_packets[2]==satellite_data[2]:
            terminations={a: True for a in self.agents}


         infos = {a: {} for a in self.agents}

        if any(terminations.values()) or all(truncations.values()):
            self.agents = []

            

        return observations, rewards, terminations, truncations, infos
def render(self):
        """Renders the environment."""
        #We will output a bar graph with the current delivery and energy expenditure
        

# Scalar values (e.g., from a list)
        values = [3, 7, 2, 5, 9]

# Optional: Labels for each bar (x-axis)
        labels = ["satellite1 Delivery Ratio","satellite2 Delivery Ratio","satellite3 Delivery Ratio","satellite1 Energy Efficiency","satellite2 Energy Efficiency","satellite3 Energy Efficiency"]

# Create the bar chart
        plt.bar(labels, values)

# Add labels and title
        plt.xlabel('KPI results')
        plt.title('System Performance')

# Show the chart
plt.show()
    # Observation space should be defined here.
    # lru_cache allows observation and action spaces to be memoized, reducing clock cycles required to get each agent's space.
    # If your spaces change over time, remove this line (disable caching).
@functools.lru_cache(maxsize=None)
def observation_space(self, agent):
    # gymnasium spaces are defined and documented here: https://gymnasium.farama.org/api/spaces/
    

# Action space should be defined here.
# If your spaces change over time, remove this line (disable caching).
@functools.lru_cache(maxsize=None)
def action_space(self, agent):
    return Discrete(1)

#Function to retrieve a single row from the master contact plan
def getRow(self,timestep):
    return self.master_contact_plan[timestep]
#Function to simply update timestep
#Where the timestep is the contact in the contact plan
def updateTimestep(self):
    self.timestep=self.timestep+1
#Get current timestep
def getTimestep(self):
    return self.timestep
#In this function we obtain the number of delivered packets and excess energy expenditure 
#If we select the given contact
def updateDeliveryandEnergy(self,agent,weather,length):
    delivered_packets=0
    excess_energy_expended=0
    for i in range(length)-1:
        random_sample=random.random()
        if random_sample> weather:
            delivered_packets=delivered_packets+1
        else:
            excess_energy_expended=excess_energy_expended+1

    return delivered_packets,excess_energy_expended
#Check if any satellites have conflicts in terms of connections
def checkActionConflict(self,timestep,actions):
    master_observation_matrix=self.master_contact_plan
    LoS_matrix=master_observation_matrix[timestep][2:4]
    #Next we must consider the actions
    conflict_matrix=[0,0,0]
    
    for j in range(2):
        if actions[j]*LoS_matrix[j]==1:
            conflict_matrix[j]=1
    penalty_matrix=[0,0,0]
    if sum(conflict_matrix)>1:
        penalty_matrix=conflict_matrix
    return penalty_matrix
    

        
        prisoner_action = actions["prisoner"]
        guard_action = actions["guard"]

        if prisoner_action == 0 and self.prisoner_x > 0:
            self.prisoner_x -= 1
        elif prisoner_action == 1 and self.prisoner_x < 6:
            self.prisoner_x += 1
        elif prisoner_action == 2 and self.prisoner_y > 0:
            self.prisoner_y -= 1
        elif prisoner_action == 3 and self.prisoner_y < 6:
            self.prisoner_y += 1

        if guard_action == 0 and self.guard_x > 0:
            self.guard_x -= 1
        elif guard_action == 1 and self.guard_x < 6:
            self.guard_x += 1
        elif guard_action == 2 and self.guard_y > 0:
            self.guard_y -= 1
        elif guard_action == 3 and self.guard_y < 6:
            self.guard_y += 1

        # Check termination conditions
        terminations = {a: False for a in self.agents}
        rewards = {a: 0 for a in self.agents}
        if self.prisoner_x == self.guard_x and self.prisoner_y == self.guard_y:
            rewards = {"prisoner": -1, "guard": 1}
            terminations = {a: True for a in self.agents}

        elif self.prisoner_x == self.escape_x and self.prisoner_y == self.escape_y:
            rewards = {"prisoner": 1, "guard": -1}
            terminations = {a: True for a in self.agents}

        # Check truncation conditions (overwrites termination conditions)
        truncations = {a: False for a in self.agents}
        if self.timestep > 100:
            rewards = {"prisoner": 0, "guard": 0}
            truncations = {"prisoner": True, "guard": True}
        self.timestep += 1

        # Get observations
        observations = {
            a: (
                self.prisoner_x + 7 * self.prisoner_y,
                self.guard_x + 7 * self.guard_y,
                self.escape_x + 7 * self.escape_y,
            )
            for a in self.agents
        }

        # Get dummy infos (not used in this example)
        infos = {a: {} for a in self.agents}

        if any(terminations.values()) or all(truncations.values()):
            self.agents = []

        return observations, rewards, terminations, truncations, infos

    def render(self):
        """Renders the environment."""
        grid = np.full((7, 7), " ")
        grid[self.prisoner_y, self.prisoner_x] = "P"
        grid[self.guard_y, self.guard_x] = "G"
        grid[self.escape_y, self.escape_x] = "E"
        print(f"{grid} \n")

    # Observation space should be defined here.
    # lru_cache allows observation and action spaces to be memoized, reducing clock cycles required to get each agent's space.
    # If your spaces change over time, remove this line (disable caching).
    @functools.lru_cache(maxsize=None)
    def observation_space(self, agent):
        # gymnasium spaces are defined and documented here: https://gymnasium.farama.org/api/spaces/
        return MultiDiscrete([7 * 7] * 3)

    # Action space should be defined here.
    # If your spaces change over time, remove this line (disable caching).
    @functools.lru_cache(maxsize=None)
    def action_space(self, agent):
        return Discrete(4)



IndentationError: unindent does not match any outer indentation level (<tokenize>, line 250)